# Open-ended EDA v2 — analysis on `data/unified_core.parquet`

Clean notebook version of the open-ended senior-DS EDA. Primary data file switched from `data/unified.parquet` (1.45M × 96) to `data/unified_core.parquet` (110k × 42, LinkedIn-only, all rows inside the Stage 9 balanced LLM-frame, with `is_control` populated in all four periods).

Hypothesis set extended to **H1–H13**. Full priors in [`../memos/priors.md`](../memos/priors.md).

**How to read this notebook:** each scan writes a figure to `eda/figures/` and a CSV to `eda/tables/`. The code here is a thin driver — all scan logic lives in `eda/scripts/{scans,core_scans}.py`. Execute top-to-bottom to regenerate all artifacts.

## 0. Setup

In [ ]:
import sys, os
from pathlib import Path

# Resolve project root
HERE = Path.cwd()
ROOT = HERE if (HERE / 'data' / 'unified_core.parquet').exists() else HERE.parents[1]
assert (ROOT / 'data' / 'unified_core.parquet').exists(), f'unified_core.parquet not found at {ROOT}'
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'eda' / 'scripts'))

import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', lambda x: f'{x:.4f}' if isinstance(x, float) else x)

from scans import AI_VOCAB_PATTERN, NEW_AI_TITLE_PATTERN, BIG_TECH_CANONICAL
from core_scans import (
    CORE_PATH, CORE_OBS_PATH, CORE_DEFAULT_FILTER, VENDOR_PATTERNS,
    rerun_s1_core, rerun_s3_core, rerun_s10_core, rerun_s11_core, rerun_sv_core,
    scan_s12, scan_s13, scan_s14, scan_s15, scan_s16, scan_s17,
)

con = duckdb.connect()
print('Environment ready.')
print(f'Core file:    data/{CORE_PATH.name}')
print(f'Core obs:     data/{CORE_OBS_PATH.name}')

## 1. Hypothesis recap (H1–H13)

Full priors in `eda/memos/priors.md`. Short form:

**Macro-narrative (from v1):**
- **H2** AI creates new job types — emergence in titles.
- **H3** Non-AI macro (Covid-binge + rate hikes + outsourcing) — Economist central claim.
- **H4** Industry spread to non-tech — Economist tech-skills-everywhere.
- **H5** Junior-first *vs* senior-restructuring within SWE.
- **H6** Big Tech vs rest — reaction differs by tier.
- **H7** SWE vs control divergence — direct SWE-specificity test.

**New in v2 (from unified_core exploration):**
- **H8** YOE floor is FALLING, not rising (counter scope-inflation).
- **H9** Dev-tool vendor labor-market leaderboard is measurable (Copilot / Claude / OpenAI / Cursor hierarchy).
- **H10** AI mention is NOT a ghost-job signal.
- **H11** Non-SWE AI-spread is niche-specific (finance + power/nuclear engineering), not broad.
- **H12** Posting survival differs by tier / AI-tag.
- **H13** Within-firm AI rewrite is real — same firm, 2024 → 2026, did they rewrite their own postings?

## 2. Phase A — corpus profile on core

In [ ]:
profile = con.execute(f"""
  SELECT source, source_platform, period,
         COUNT(*) AS n,
         SUM(CASE WHEN is_swe THEN 1 ELSE 0 END) AS swe,
         SUM(CASE WHEN is_swe_adjacent THEN 1 ELSE 0 END) AS adjacent,
         SUM(CASE WHEN is_control THEN 1 ELSE 0 END) AS control
  FROM '{CORE_PATH}'
  GROUP BY 1,2,3 ORDER BY 1,2,3
""").df()
profile['total_check'] = profile['swe'] + profile['adjacent'] + profile['control']
print('Core row count:', profile['n'].sum())
profile

In [ ]:
coverage = con.execute(f"""
  SELECT period, analysis_group, llm_classification_coverage,
         llm_extraction_coverage, COUNT(*) AS n
  FROM '{CORE_PATH}'
  GROUP BY 1,2,3,4 ORDER BY 1,2,3,4
""").df()
coverage_pivot = coverage.pivot_table(
    index=['period','analysis_group'],
    columns='llm_classification_coverage',
    values='n', fill_value=0, aggfunc='sum'
)
coverage_pivot

## 3. Phase B — re-run finalists on core

Five v1 finalists re-executed against unified_core. Writes to `eda/tables/{S1,S3,S10,S11,Sv}_core_*.csv` and matching PNGs.

In [ ]:
s1 = rerun_s1_core(con); s1

In [ ]:
s3 = rerun_s3_core(con); s3

In [ ]:
s10 = rerun_s10_core(con); s10

In [ ]:
s11 = rerun_s11_core(con); s11

In [ ]:
sv = rerun_sv_core(con); sv

## 4. Phase B — new scans S12–S17 (H8–H13)

### S12 (H8) — YOE floor trajectory

In [ ]:
s12 = scan_s12(con)
display(s12)

**Read:** junior mean YOE 2.01 → 1.23 (mean) and 2 → 1 (median) across 2024→2026. Senior median dropped 6 → 5. Classic scope-inflation at the YOE bar is falsified on LLM-YOE.

### S13 (H9) — Vendor labor-market leaderboard

In [ ]:
s13 = scan_s13(con)
s13

In [ ]:
# Stand-alone 2026-04 leaderboard
pd.read_csv('eda/tables/S13_vendor_leaderboard_2026_04.csv')

**Read:** in 2026-04 SWE postings, Copilot leads at 4.25%, Claude second at 3.83%, OpenAI third at 3.63%. Cursor emerged from ~0% to 2.17% in two years. ChatGPT as a brand is PLATEAUING (0.82% → 0.70%) while Claude/OpenAI/Anthropic continue climbing.

### S14 (H10) — AI-mention × ghost rate

In [ ]:
s14 = scan_s14(con)
s14

**Read:** AI-mentioning SWE postings are *slightly less* inflated than non-AI postings (4.5% vs 5.6% in 2026-04). The 'AI buzzword = ghost job' narrative is not supported by our LLM ghost assessment.

### S15 (H11) — Control AI-rise occupational drivers

In [ ]:
s15 = scan_s15(con)
# Rank families by 2026-04 AI-rate
latest = s15[s15['period']=='2026-04'].sort_values('ai_rate', ascending=False)
latest

**Read:** the control-tier AI rise concentrates in finance/accounting and electrical/nuclear engineering. Nursing, HR, sales, marketing are essentially untouched. Reframes H4 from 'tech skills everywhere' to 'AI language is reaching finance and utility engineering; service-occupation control still absent.'

### S16 (H12) — Posting survival

In [ ]:
s16 = scan_s16(con)
s16

**Read:** in 2026-03 (month with complete observation window), AI-mentioning SWE postings stay alive 0.9 days longer on average (5.27 vs 4.36 days). Control AI-mentioning postings show a larger gap (7.5 vs 2.6 days) but the sample is small. Small positive signal consistent with either higher demand or slower filling. 2026-04 is still ongoing and not directly comparable.

### S17 (H13) — Within-firm AI rewrite overlap panel

In [ ]:
s17 = scan_s17(con)
print('Panel size:', len(s17), 'companies')
print(f"Mean delta 2026−2024: {s17['delta'].mean()*100:.2f}pp")
print(f"Median delta:         {s17['delta'].median()*100:.2f}pp")
print(f"% companies with rise:    {(s17['delta']>0).mean()*100:.1f}%")
print(f"% with rise >10pp:        {(s17['delta']>0.10).mean()*100:.1f}%")
print(f"% with rise >20pp:        {(s17['delta']>0.20).mean()*100:.1f}%")
print()
print('Top 15 risers:')
s17.sort_values('delta', ascending=False).head(15)

**Read:** on the 292-company overlap panel (companies with ≥5 SWE postings in BOTH 2024 Kaggle and 2026 scraped), the *same companies* rewrote their postings. Mean within-firm delta = **+19.4pp**; 75% of companies rose; 61% rose more than 10pp; 39% rose more than 20pp. Microsoft +51pp, Wells Fargo +45pp, Amazon +36pp. Defense firms (Raytheon, Northrop Grumman) flat or near-flat.

**This rules out between-firm composition as the sole driver of the 2024→2026 AI rewrite.** The rewrite is happening INSIDE the same firms.

## 5. Headlines — v2 verdict table

In [ ]:
verdicts = pd.DataFrame([
    ('H2  new AI job types',          'supported',
     'S3 new-AI-title share 1.6% → 8.3% (5×)'),
    ('H3  non-AI macro (content)',    'against',
     'S11 SWE-specificity rules out economy-wide cause'),
    ('H4  industry spread',           'NOT on LinkedIn',
     'S8 non-tech share flat ~55% 2024→2026'),
    ('H5  junior-first vs senior',    '(b) senior consistent; (a) falsified',
     'S1 AI-vocab uniform across levels; S12 junior YOE FELL'),
    ('H6  Big Tech vs rest',          'split',
     'S10 BT AI 44% vs rest 27% (diff 17pp); BT SHARE ROSE not fell'),
    ('H7  SWE-vs-control',            'strongly supports SWE-specificity',
     'S11 on balanced core with real 2024 control baseline'),
    ('H8  YOE floor falling',         'supported',
     'S12 junior mean YOE 2.01 → 1.23, median 2 → 1'),
    ('H9  vendor leaderboard',        'supported (Copilot > Claude > OpenAI > Cursor)',
     'S13 2026-04: Copilot 4.25%, Claude 3.83%, OpenAI 3.63%'),
    ('H10 AI mention ≠ ghost job',    'supported',
     'S14 AI inflated rate 4.5% vs non-AI 5.6%'),
    ('H11 control spread is niche',   'supported',
     'S15 finance + electrical/nuclear dominate; service occupations flat'),
    ('H12 posting survival',          'directional only',
     'S16 AI postings live 0.9 days longer in 2026-03 (small n for control)'),
    ('H13 within-firm AI rewrite',    'strongly supported',
     'S17 mean +19.4pp, 75% of 292 companies rose'),
], columns=['hypothesis', 'verdict', 'primary evidence'])
verdicts

## 6. Recommended next research moves (ranked)

1. **Big-Tech-stratified RQ1 paper.** The 17pp BT-vs-rest AI-density gap (S10) and the within-firm rise (S17) are strongest at Big Tech; pair with named-firm layoff timelines to test whether AI-density co-moves with announcements.
2. **Vendor-specificity paper (S13).** Nobody has published a labor-demand vendor-share table for dev tools. The hierarchy (Copilot > Claude > OpenAI > Cursor, with ChatGPT plateauing) is entirely new.
3. **YOE-floor inversion paper (S12).** Rule-based YOE in v8 rose compositionally; LLM-YOE declines across all seniority buckets. Contradicts the popular 'scope inflation' narrative. Worth calling out explicitly with the methodological note on rule-vs-LLM YOE.
4. **Within-firm rewrite mechanism (S17) → RQ4 interviews.** v8 established cross-seniority rewriting; S17 localizes it to the within-firm channel. Interview script can ask: 'who inside your firm rewrote the JDs between 2024 and 2026?' Connects employer-side content to the ghost-vs-real debate (RQ4).

## 7. Limitations (v2)

- `unified_core.parquet` is LinkedIn-only by construction — no Indeed sensitivity.
- Control rows in 2024 come from Kaggle (asaniczka + arshkon) matched against Stage-5 `is_control` classification; the 2026 control rows come from explicit scraper query tiers. The *classification criterion* is consistent but the *recruitment surface* differs.
- `unified_core_observations.parquet` has no scrape-cadence for Kaggle sources — S16 posting-survival is 2026 only.
- Vendor-specific regexes match bare words ('cursor', 'claude', 'gemini') — some false positives from unrelated uses (cursor as a UI element, claude as a person's name). Rates should be read directionally.
- Within-firm panel (S17) requires ≥5 SWE postings in both periods; smaller firms are excluded. 292 firms is meaningful but not exhaustive.
- H8 YOE finding depends on `yoe_min_years_llm` coverage (~80% of in-frame SWE rows). Selection bias possible: rows where LLM refused to extract YOE might differ systematically.
- No significance tests. Descriptive only.